# B3-J2 Atelier 1 — Baseline régression : Prix immobilier

**Durée** : 1h30 (11h15-12h45)

## Objectifs

À la fin de cet atelier, vous savez :

1. Charger un dataset tabulaire et faire une **EDA rapide**
2. Séparer **features X** et **cible y**, puis splitter en **train/val/test**
3. Construire un **ColumnTransformer** (OneHotEncoder + StandardScaler)
4. Encapsuler le tout dans un **Pipeline Scikit-learn**
5. Entraîner un modèle baseline (**LinearRegression**) et lire ses scores (MAE, RMSE, R²)

**Dataset** : California Housing (prix immobiliers californiens, fourni par sklearn).

**Stack** : Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn.

## Pourquoi un baseline simple ?

> *"Tu ne sais pas si ton XGBoost est bon tant que tu n'as pas mesuré ce que fait une LinearRegression sur les mêmes données."*

Un baseline c'est :

- **Un point de comparaison** : tout modèle plus complexe doit faire mieux, sinon c'est inutile.
- **Une sanity check du pipeline** : si la LinearRegression est cassée, le RandomForest le sera aussi.
- **Un investissement faible** : 5 lignes de code, 2 secondes d'entraînement.

Aujourd'hui on construit ce baseline **proprement**, dans un Pipeline réutilisable demain pour 3 modèles.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Imports OK")

## Étape 1/8 — Charger le dataset

On utilise **California Housing** depuis `sklearn.datasets`. C'est un dataset de prix médian de maisons par district californien (recensement 1990).

- **Cible (y)** : `MedHouseVal` — prix médian en centaines de milliers de dollars.
- **Features (X)** : 8 variables numériques (revenu médian, âge médian des maisons, nombre de pièces, latitude/longitude…).

Note : on remplace Boston Housing (déprécié pour problème éthique) par California Housing.

In [ ]:
# Charger via sklearn → DataFrame Pandas
data = fetch_california_housing(as_frame=True)
df = data.frame  # DataFrame complet (features + target)

print("Description du dataset :")
print(data.DESCR[:1500])

In [ ]:
# Ajoutons une feature catégorielle simulée pour rendre l'atelier plus réaliste
# (en prod tu aurais souvent du catégoriel : type de bien, ville, etc.)
np.random.seed(42)
df["OceanProximity"] = np.random.choice(
    ["NEAR_BAY", "INLAND", "NEAR_OCEAN", "ISLAND"],
    size=len(df),
    p=[0.25, 0.45, 0.25, 0.05],
)

df.head()

## Étape 2/8 — EDA rapide

Avant de modéliser, on regarde **toujours** :

- Forme du dataset (`.shape`)
- Types et valeurs manquantes (`.info()`)
- Statistiques descriptives (`.describe()`)
- Corrélations entre features
- Distribution de la cible

In [ ]:
print("Shape :", df.shape)
print("\n--- Info ---")
df.info()

In [ ]:
df.describe()

In [ ]:
# Heatmap de corrélations (numérique uniquement)
corr = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Corrélations features numériques")
plt.show()

In [ ]:
# Distribution de la cible y
# À compléter : tracer un histogramme de df["MedHouseVal"]
plt.figure(figsize=(8, 4))
# À compléter : utiliser sns.histplot ou plt.hist sur df["MedHouseVal"]
sns.histplot(df["MedHouseVal"], bins=50, kde=True)
plt.title("Distribution du prix médian (cible)")
plt.xlabel("MedHouseVal (×100k $)")
plt.show()

# À compléter : calculer le skew avec .skew() et le commenter en print
skew = df["MedHouseVal"].skew()
print(f"Skew = {skew:.3f}  →  > 0 : queue à droite (quelques maisons très chères)")

**Observation** : la cible est plafonnée à 5.0 (artefact du dataset original). On va l'ignorer pour le baseline mais en prod ça mériterait un traitement explicite (clipping conscient, modèle censuré...).

## Étape 3/8 — Identifier cible y et features X

Convention :
- `X` = tout sauf la cible
- `y` = la colonne à prédire

In [ ]:
TARGET = "MedHouseVal"

# À compléter : créer X (toutes les colonnes SAUF TARGET) et y (la colonne TARGET)
X = df.drop(columns=[TARGET])
y = df[TARGET]  # À compléter

print("X shape :", X.shape)
print("y shape :", y.shape)
print("\nColonnes X :", list(X.columns))

## Étape 4/8 — Train/val/test split

Pourquoi **trois** sets ?

- **Train** (~70%) : pour `.fit()` — le modèle voit ces données.
- **Val** (~15%) : pour comparer différentes versions du modèle (features, hyperparamètres).
- **Test** (~15%) : ouvert **une seule fois à la fin** — c'est ton vrai score honnête.

Règle d'or : **`random_state` fixé** pour la reproductibilité.

Astuce : on fait deux splits successifs (train+val vs test, puis train vs val).

In [ ]:
# Split 1 : sortir le test set (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# Split 2 : séparer train (70% du total) et val (15% du total)
# À compléter : utiliser train_test_split sur X_trainval / y_trainval
#              test_size doit être tel que val = 15% du total initial
#              indice : 0.15 / 0.85 ≈ 0.176
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15 / 0.85, random_state=42
)

print(f"Train : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X):.0%})")
print(f"Val   : {X_val.shape[0]} lignes ({X_val.shape[0]/len(X):.0%})")
print(f"Test  : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X):.0%})")

## Étape 5/8 — ColumnTransformer (preprocessing)

On a **2 types de features** :

- **Numériques** → besoin d'être mises à la même échelle (`StandardScaler`).
- **Catégorielles** (`OceanProximity`) → besoin d'être encodées en numérique (`OneHotEncoder`).

Le `ColumnTransformer` applique le bon preprocessing à chaque type de colonne, **en respectant le split** (le scaler apprend uniquement sur le train).

In [ ]:
# Identifier les colonnes par type
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numériques  :", numeric_features)
print("Catégorielles :", categorical_features)

# À compléter : créer le ColumnTransformer
# Indice : ColumnTransformer([("nom", transformer, colonnes), ...])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        # À compléter : ajouter une étape "cat" avec OneHotEncoder(handle_unknown="ignore") sur categorical_features
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

print("\nPreprocessor créé :")
print(preprocessor)

## Étape 6/8 — Pipeline + LinearRegression

Le `Pipeline` chaîne preprocessing + modèle dans un seul objet :

1. `.fit(X_train, y_train)` → fit preprocessor sur train, puis fit modèle sur train transformé.
2. `.predict(X_val)` → transform val avec le preprocessor déjà fitté, puis predict.

**Avantage clé** : zéro data leakage du val/test vers le train, automatiquement.

In [ ]:
# À compléter : construire le Pipeline avec preprocessor + LinearRegression
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

# Entraînement
model.fit(X_train, y_train)

print("Modèle entraîné OK")

## Étape 7/8 — Évaluation

Trois métriques de régression à connaître :

| Métrique | Lecture | Quand l'utiliser |
|---|---|---|
| **MAE** (Mean Absolute Error) | Erreur moyenne en unité de y | Robuste aux outliers, interprétable |
| **RMSE** (Root Mean Squared Error) | Pénalise les grosses erreurs | Si les grosses erreurs sont coûteuses |
| **R²** | Part de variance expliquée (1 = parfait, 0 = aussi nul que la moyenne, <0 = pire que la moyenne !) | Interprétation pédagogique |

In [ ]:
# À compléter : prédictions sur le set de validation
y_pred_val = model.predict(X_val)

# À compléter : calculer MAE, RMSE, R²
mae = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2 = r2_score(y_val, y_pred_val)

print("=== Scores sur le set de validation ===")
print(f"MAE   : {mae:.3f}  (≈ {mae*100:.1f} k$ d'erreur moyenne)")
print(f"RMSE  : {rmse:.3f}")
print(f"R²    : {r2:.3f}")
print(f"\n.score() raccourci : {model.score(X_val, y_val):.3f}  (= R² par défaut)")

## Étape 8/8 — Visualiser prédictions vs réel

Un scatter plot `y_pred` vs `y_val` est **le** graphique à toujours produire :

- Si le modèle était parfait → tous les points sur la diagonale `y = x`.
- Les écarts à la diagonale = erreurs.
- On voit immédiatement les biais (sous/sur-estimation systématique, plafonnement, etc.).

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_val, y_pred_val, alpha=0.3, s=10)

# Ligne identité y = x
lims = [min(y_val.min(), y_pred_val.min()), max(y_val.max(), y_pred_val.max())]
plt.plot(lims, lims, "r--", label="y = x (prédiction parfaite)")

plt.xlabel("Prix réel (val)")
plt.ylabel("Prix prédit")
plt.title(f"Prédictions vs réel — R² = {r2:.3f}")
plt.legend()
plt.axis("equal")
plt.show()

## Récap — ce qu'on vient de faire

- **Chargé** un dataset tabulaire (California Housing) en DataFrame Pandas.
- **Exploré** rapidement (`.info`, `.describe`, corrélations, distribution cible).
- **Séparé** X / y puis splitté en train/val/test avec `random_state=42`.
- **Construit** un `ColumnTransformer` (Scaler numérique + OneHot catégoriel) encapsulé dans un `Pipeline` avec `LinearRegression`.
- **Évalué** avec MAE, RMSE, R² sur le set de validation + scatter plot prédictions vs réel.

On a un **baseline reproductible et propre** qui servira de point de comparaison cet après-midi.

## Discussion — Le R² est-il bon ? Comment savoir ?

Quelques questions à se poser :

1. **R² ≈ 0.6** → pas mauvais, pas excellent. Référentiel ?
2. **Le score val dépend-il du split ?** Si je change `random_state`, j'obtiens combien ? Comment savoir si le score est **stable** ?
3. **Une LinearRegression capture-t-elle vraiment les interactions entre features ?**
4. **Comment comparer rigoureusement plusieurs modèles ?**

→ **Réponses dans l'Atelier 2 cet après-midi** : cross-validation, comparaison RandomForest / GradientBoosting, GridSearchCV, et feature importance.